In [ ]:
# Forecast of products demand based on M5 dataset

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from catboost import CatBoostRegressor
from src.data.catboost_config import run_catboost_experiment
import gc
from sklearn.metrics import mean_absolute_error, mean_squared_error
from src.data.evaluate_model import evaluate_model

from src.data.load_data import load_data, prepare_calendar
from src.data.build_features import build_features

from src.data.load_data import load_data, prepare_calendar
from src.data.split_data import split_data

In [3]:
data_path = "../data/m5"


sales, calendar, prices = load_data(data_path)
calendar = prepare_calendar(calendar)

df = build_features(
    sales=sales,
    calendar=calendar,
    prices=prices,
)
# nonzero_days_14 — число дней с ненулевыми продажами за последние 14 дней.
# nonzero_rate_14 — доля дней с ненулевыми продажами.
# avg_positive_demand_14 — средний объём продаж в дни с положительным спросом.
# adi_14 — средний интервал между днями с ненулевыми продажами.
# demand_regime_14 — классификация спроса на intermittent, regular или unknown.
print(df.shape)
print(df.head())
print(df.columns)

(58327370, 40)
       item_id  dept_id cat_id store_id state_id    d  sales       date  \
0  FOODS_1_001  FOODS_1  FOODS     CA_1       CA  d_1      3 2011-01-29   
1  FOODS_1_001  FOODS_1  FOODS     CA_1       CA  d_2      0 2011-01-30   
2  FOODS_1_001  FOODS_1  FOODS     CA_1       CA  d_3      0 2011-01-31   
3  FOODS_1_001  FOODS_1  FOODS     CA_1       CA  d_4      1 2011-02-01   
4  FOODS_1_001  FOODS_1  FOODS     CA_1       CA  d_5      4 2011-02-02   

   wm_yr_wk  sell_price  ...  history_days_14  nonzero_rate_14  \
0     11101         2.0  ...              0.0              NaN   
1     11101         2.0  ...              1.0         1.000000   
2     11101         2.0  ...              2.0         0.500000   
3     11101         2.0  ...              3.0         0.333333   
4     11101         2.0  ...              4.0         0.500000   

   avg_positive_demand_14 adi_14 demand_regime_14 price_lag_1 price_median_7  \
0                     NaN    NaN          unknown        

## We finished with features taken into account. Starting to build seasonal naive baseline

In [4]:
# Определяем категориальные признаки
cat_features = [
    "item_id",
    "dept_id",
    "store_id",
    "event_name_1",
    "event_type_1",
    "demand_regime_14",
]

# Определяем числовые признаки
num_features = [
    "wday",
    "month",
    "year",
    "week_of_year",
    "day_of_month",
    "snap_CA",
    "snap_TX",
    "snap_WI",
    "sell_price",
    "lag_1",
    "lag_7",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28",
    "rolling_std_28",
    "price_lag_1",
    "price_median_7",
    "price_median_28",
    "price_ratio_1",
    "price_ratio_28",
    "nonzero_days_14",
    "history_days_14",
    "nonzero_rate_14",
    "avg_positive_demand_14",
    "adi_14",
]

feature_cols = cat_features + num_features

# Проверяем соответствие признаков датафрейму
df_columns = set(df.columns)
feature_set = set(feature_cols)


missing_features = sorted(feature_set - df_columns)
extra_columns = sorted(
    df_columns - feature_set
    - {
        "sales", "date", "d", "wm_yr_wk",
        "cat_id", "state_id", "event_name_2", "event_type_2",
    }
)

if missing_features or extra_columns:
    if missing_features:
        print("Не хватает признаков:", missing_features)
    if extra_columns:
        print("Лишние колонки в датафрейме:", extra_columns)

    raise ValueError("Проверь состав признаков перед обучением модели.")

print("Все признаки соответствуют датафрейму. Можно продолжать.")

Все признаки соответствуют датафрейму. Можно продолжать.


In [ ]:
from catboost import CatBoostRegressor

baseline_config = {
    "iterations": 500,
    "learning_rate": 0.05,
    "depth": 8,
    "l2_leaf_reg": 3,
}

# Даты окончания тестовых периодов с шагом 28 дней
df_dates = pd.to_datetime(df["date"])
first_test_end = df_dates.min() + pd.Timedelta(days=172)
last_date = df_dates.max()

fold_ends = pd.date_range(
    start=first_test_end,
    end=last_date,
    freq="28D",
)

backtest_results = []

for fold_num, fold_end in enumerate(fold_ends, start=1):
    train_df, validation_df, test_df, train_window = split_data(
        df,
        end_date=fold_end,
    )

    # Проверяем, что все нужные периоды непустые
    if (
        train_window.empty
        or validation_df.empty
        or test_df.empty
    ):
        continue

    X_train = train_window[feature_cols]
    y_train = train_window["sales"]

    X_val = validation_df[feature_cols]
    y_val = validation_df["sales"]

    X_test = test_df[feature_cols]
    y_test = test_df["sales"]

    model = CatBoostRegressor(
        **baseline_config,
        loss_function="RMSE",
        eval_metric="RMSE",
        random_seed=42,
        verbose=False,
        allow_writing_files=False,
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
        eval_set=(X_val, y_val),
        early_stopping_rounds=100,
        verbose=False,
    )

    # Оцениваем именно на следующем тестовом периоде
    pred_test = model.predict(X_test)

    result = evaluate_model(
        f"backtest_fold_{fold_num}",
        y_test,
        pred_test,
    )

    result["fold"] = fold_num
    result["test_start"] = test_df["date"].min()
    result["test_end"] = test_df["date"].max()
    result["train_start"] = train_window["date"].min()
    result["train_end"] = train_window["date"].max()

    backtest_results.append(result)

backtest_df = pd.DataFrame(backtest_results)

print(backtest_df)

In [6]:
# Подготавливаем данные для модели
X_train = train_window[feature_cols]
y_train = train_window["sales"]

X_val = validation_df[feature_cols]
y_val = validation_df["sales"]

X_test = test_df[feature_cols]
y_test = test_df["sales"]

In [13]:
# Seasonal Naive baseline: продажи за 7 дней до прогнозируемой даты
val_pred_baseline = validation_df["lag_7"].to_numpy()

evaluate_model(
    "Baseline: lag_7",
    y_val,
    val_pred_baseline,
)

Model: Baseline: lag_7
MAE:   1.2055
RMSE:  2.7634
WAPE:  87.14%


{'model': 'Baseline: lag_7',
 'MAE': 1.2054784203685482,
 'RMSE': np.float64(2.763418737969946),
 'WAPE_%': np.float64(87.14252095173198)}

In [ ]:
### We prepared everything for building a model, so let's go


from catboost import CatBoostRegressor
import numpy as np
import pandas as pd

# Quantiles we want to predict
quantiles = [0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80]

models = {}
forecasts = {}

for q in quantiles:
    model = CatBoostRegressor(
        loss_function=f'Quantile:alpha={q}',
        eval_metric='MAE',
        iterations=500,
        learning_rate=0.05,
        depth=8,
        random_seed=42,
        verbose=100
    )

    model.fit(
        X_train_sample,
        y_train_sample,
        cat_features=cat_features
    )

    models[q] = model
    forecasts[q] = model.predict(X_validation)

# Combine all quantile forecasts into one dataframe
validation_features = validation_features.copy()

for q in quantiles:
    validation_features[f'forecast_p{int(q * 100)}'] = forecasts[q]

validation_features[
    ['item_id', 'store_id', 'target_date', 'horizon',
     'forecast_p20', 'forecast_p30', 'forecast_p40',
     'forecast_p50', 'forecast_p60', 'forecast_p70',
     'forecast_p80']
].head(10)


### Now we combine the prediction and the actuals
def pinball_loss(y_true, y_pred, q):
    errors = y_true - y_pred
    return np.maximum(q * errors, (q - 1) * errors).mean()


# Join actual sales to forecasts
validation_result = validation_features[
    ['item_id', 'store_id', 'target_date', 'horizon'] +
    [f'forecast_p{int(q * 100)}' for q in quantiles]
].merge(
    validation_df[['item_id', 'store_id', 'date', 'sales']],
    how='left',
    left_on=['item_id', 'store_id', 'target_date'],
    right_on=['item_id', 'store_id', 'date']
).rename(columns={'sales': 'actual'})

# Evaluate each quantile
results = []

for q in quantiles:
    col = f'forecast_p{int(q * 100)}'

    results.append({
        'quantile': f'P{int(q * 100)}',
        'MAE': (validation_result['actual'] -
                validation_result[col]).abs().mean(),
        'Pinball loss': pinball_loss(
            validation_result['actual'],
            validation_result[col],
            q
        ),
        'Coverage': (
            validation_result['actual'] <=
            validation_result[col]
        ).mean()
    })

metrics = pd.DataFrame(results)
print(metrics.round(3))


In [ ]:
### Now we are looking for the best parameter using grid iterations*depth

iterations_list = [500, 1000, 1500]
depth_list = [6, 8, 10]

results = []

for iterations in iterations_list:
    for depth in depth_list:

        print(f'\nTraining: iterations={iterations}, depth={depth}')

        model = CatBoostRegressor(
            loss_function='RMSE',
            eval_metric='MAE',
            iterations=iterations,
            learning_rate=0.05,
            depth=depth,
            random_seed=42,
            verbose=False
        )

        model.fit(
            X_train_sample,
            y_train_sample,
            cat_features=cat_features
        )

        forecast = model.predict(X_validation)

        mae = (
            validation_result['actual'] - forecast
        ).abs().mean()

        rmse = np.sqrt(
            ((validation_result['actual'] - forecast) ** 2).mean()
        )

        results.append({
            'iterations': iterations,
            'depth': depth,
            'MAE': mae,
            'RMSE': rmse
        })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values('MAE')

print(results_df)
# Training: iterations=1500, depth=10
#    iterations  depth       MAE      RMSE
# 3        1000      6  1.063566  2.282704
# 6        1500      6  1.065099  2.288695
# 0         500      6  1.065220  2.282283
# 2         500     10  1.066018  2.292545
# 1         500      8  1.068164  2.284326
# 5        1000     10  1.070801  2.304752
# 7        1500      8  1.073315  2.296634
# 8        1500     10  1.074646  2.314555
# 4        1000      8  1.075338  2.289589